# AgentComet — Complete Feature Guide

This notebook demonstrates **all** features of the AgentComet SDK:

1. **Tools** — `@tool` decorator + builtin tools
2. **Agent Class** — Custom agents with `setup()`
3. **Memory & Conversations** — Auto-saved chat history + key-value store
4. **State Persistence** — Named checkpoints with rollback
5. **UAF Export & Load** — Portable agents that remember everything
6. **LLM Providers** — Ollama, OpenAI, Gemini, Anthropic, etc.

**Prerequisites:**
```bash
pip install uaf_compiler pyyaml requests
pip install -e .  # Install agentcomet
```

---

## Setup

In [ ]:
import os
import shutil
from agentcomet import Agent, create_agent, load_agent
from agentcomet.models import Ollama
from agentcomet.tools import tool

llm = Ollama(model="gemma3:4b")
print("LLM ready:", llm.model)

---

## 1. Tools

The `@tool` decorator converts any function into a `ToolSpec` with auto-extracted name, description, and JSON schema.

In [ ]:
@tool
def multiply(a: int, b: int) -> int:
    """Multiplies two numbers together."""
    return a * b

@tool
def calculator(expression: str) -> str:
    """Evaluates a mathematical expression."""
    return str(eval(expression))

print("Name:", multiply.name)
print("Description:", multiply.description)
print("Schema:", multiply.schema)

In [ ]:
# Builtin tools
from agentcomet.tools import read, write

print("Builtin:", read.name, "-", read.description)
print("Builtin:", write.name, "-", write.description)

---

## 2. Agent Class

Subclass `Agent`, configure in `setup()`, pass LLM at instantiation.

In [ ]:
class MyAssistant(Agent):
    def setup(self):
        self.name = "assistant"
        self.description = "Personal assistant that remembers you"
        self.author = "Vaibhav"
        self.add_tools(multiply, calculator)

agent = MyAssistant(llm=llm)
print(agent.run("Hello! What can you do?"))

---

## 3. Memory & Conversations

Every agent has `self.memory`. When you call `agent.run()`, the conversation is **automatically saved** to `memory["messages"]`.

### 3A. Conversational memory — share info, agent remembers

In [ ]:
# Share personal info with the agent
print(agent.run("Hi, my name is Vaibhav and my phone number is 9876543210"))

In [ ]:
print(agent.run("I work at AgentComet as a developer"))

In [ ]:
print(agent.run("Remember: project deadline is March 15th"))

In [ ]:
# Now ask — the agent recalls from conversation history
print(agent.run("What is my name?"))

In [ ]:
print(agent.run("What is my phone number?"))

In [ ]:
# Check stored messages
msgs = agent.memory.get("messages", [])
print(f"\nConversation has {len(msgs)} messages:")
for msg in msgs:
    print(f"  [{msg['role']}] {msg['text'][:80]}")

### 3B. Key-value memory — store structured data

In [ ]:
# You can also store arbitrary key-value data
agent.memory.save("system_prompt", "You are a helpful personal assistant.")
agent.memory.save("preferences", {"theme": "dark", "lang": "en"})
agent.memory.save("notes", [
    "User prefers concise answers",
    "Deadline is March 15th"
])

print("All keys:", agent.memory.keys())
print("System prompt:", agent.memory.get("system_prompt"))
print("Notes:", agent.memory.get("notes"))
print()
print(agent.memory)  # Memory(N keys: [...])

---

## 4. State Persistence

Save memory snapshots — including full conversation history. Rollback anytime.

In [ ]:
# Save current state (with all messages + data)
hash1 = agent.save_state()  # auto-hash
agent.save_state("after-intro")  # friendly name

In [ ]:
# Continue chatting — update info
print(agent.run("Actually, the deadline moved to March 20th"))
agent.memory.save("notes", ["Deadline updated to March 20th"])

agent.save_state("updated-deadline")

In [ ]:
# View all checkpoints
agent.show_states()

In [ ]:
# Rollback to before deadline change
agent.load_state("after-intro")
print("After rollback:")
print("  Notes:", agent.memory.get("notes"))
print("  Messages:", len(agent.memory.get("messages", [])), "messages")

# Ask about deadline — should reflect the ORIGINAL date
print(agent.run("When is the project deadline?"))

---

## 5. UAF Export & Load — Agent Remembers After Reload

Export to `.uaf` — conversation + memory auto-packed.  
Load later — agent picks up right where you left off.

In [ ]:
# Reset to latest state (has all the info)
agent.load_state("updated-deadline")

# Export — everything auto-packed
agent.export("my_assistant.uaf")
print("Exported!")

In [ ]:
# Simulate a fresh session — load from file
loaded = load_agent("my_assistant.uaf")

print("Loaded agent type:", type(loaded))
print(f"Restored {len(loaded.memory.get('messages', []))} messages")
print("Restored notes:", loaded.memory.get("notes"))
print()

# Ask the loaded agent about info from BEFORE the export
print("--- Asking loaded agent about stored info ---")
print(loaded.run("What is my name?"))
print(loaded.run("What is my phone number?"))
print(loaded.run("When is the project deadline?"))

In [ ]:
# Cleanup
os.remove("my_assistant.uaf")

---

## 6. LLM Providers

Pass LLM instances directly to agents.

In [ ]:
from agentcomet.models import Ollama, OpenAIChat, Gemini, Anthropic, OpenRouter, Perplexity

# Ollama (local)
ollama = Ollama(model="gemma3:4b")
result = ollama.generate("What is the capital of France?")
print("Direct call:", result[:100])

# Other providers (require API keys)
# openai = OpenAIChat(model="gpt-4o")
# gemini = Gemini(model="gemini-1.5-flash")
# claude = Anthropic(model="claude-3-5-sonnet")

---

## Cleanup

In [ ]:
if os.path.exists(".agentcomet"):
    shutil.rmtree(".agentcomet")
    print("Cleaned up .agentcomet/")

---

## Summary

| Feature | API | Description |
|---------|-----|-------------|
| **Tools** | `@tool` | Auto ToolSpec from functions |
| **Agent** | `class MyAgent(Agent)` | Full control via `setup()` |
| **Declarative** | `create_agent(...)` | One-liner agents |
| **Auto Memory** | `agent.run("...")` | Chat auto-saved to messages |
| **Key-Value** | `memory.save(k, v)` | Store anything |
| **Save State** | `save_state("name")` | Named or hashed checkpoint |
| **Load State** | `load_state("name")` | Rollback to any checkpoint |
| **Export** | `agent.export("file.uaf")` | Memory auto-packed |
| **Load** | `load_agent("file.uaf")` | Agent remembers everything |
| **LLM** | `Ollama(model=...)` | 6+ providers, pass directly |